# Single Pendulum: Energy Swing-Up with PID or LQR

This notebook separates the nonlinear swing-up controller from the local upright
stabilizer. Choose PID or LQR in one selector cell. The default actuator can supply
100 N, while energy pumping is limited to 20 N. All gains and switching thresholds
are exposed for tuning.


## 1. Runtime setup
The simulation is headless and self-contained.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
    drive.mount("/content/drive")

IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if IN_COLAB or IN_KAGGLE:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--upgrade", "-q",
        "mujoco>=3.2", "gymnasium>=1.0", "pandas>=2.0", "matplotlib>=3.8",
        "imageio>=2.34", "imageio-ffmpeg>=0.5", "scipy>=1.11",
    ], check=True)

# Native MuJoCo dynamics are CPU-based; EGL uses the hosted NVIDIA GPU for video rendering.
os.environ.setdefault("MUJOCO_GL", "egl")
print("Runtime:", "Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "local")


## 2. Model, tuning, and saved artifacts


In [ ]:
import json
import mujoco
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.linalg import solve_continuous_are

OUTPUT_DIR = Path("/content/drive/MyDrive/ProjectsRuns/TIPy/runs/single/classical/run-001") if IN_COLAB else Path.cwd() / "classical-run"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TRAJECTORY_PATH, CONFIG_PATH, DASHBOARD_PATH = OUTPUT_DIR/"trajectory.csv", OUTPUT_DIR/"tuned_config.json", OUTPUT_DIR/"dashboard.png"

DT, CART_MASS, POLE_MASS, POLE_LENGTH, GRAVITY = .005, 2.0, .7, .6, 9.81
RAIL_LIMIT, ACTUATOR_LIMIT, SWING_LIMIT = 2.2, 100.0, 20.0
ENERGY_GAIN, CENTER_KP, CENTER_KD, STARTUP_FORCE = 20.0, 8.0, 4.0, 3.0
PID_GAINS = {"position_kp":.8,"position_ki":.05,"position_kd":1.2,
             "angle_kp":90.,"angle_ki":2.,"angle_kd":18.,"max_tilt":.25}
LQR_Q = np.diag([20., 200., 5., 20.]); LQR_R = np.array([[.1]])
CAPTURE_ANGLE, CAPTURE_SPEED, CAPTURE_DWELL = .35, 3.0, .10
FALLBACK_ANGLE, CAPTURE_X, CAPTURE_DX, CATCH_SECONDS = .70, 1.5, 2.0, .8
SIMULATION_SECONDS = 15.0

MODEL_XML = r"""
<mujoco model="cartpole_single">
  <compiler angle="radian" autolimits="true" inertiafromgeom="false" />
  <option timestep="0.005" gravity="0 0 -9.81" integrator="RK4" />
  <default>
    <geom contype="0" conaffinity="0" />
  </default>
  <visual>
    <headlight diffuse="0.7 0.7 0.7" ambient="0.3 0.3 0.3" specular="0.1 0.1 0.1" />
    <rgba haze="0.15 0.2 0.25 1" />
  </visual>
  <asset>
    <material name="floor_mat" rgba="0.16 0.22 0.28 1" />
    <material name="metal_mat" rgba="0.65 0.68 0.72 1" />
    <material name="cart_mat" rgba="0.82 0.18 0.18 1" />
    <material name="pole_mat" rgba="0.2 0.75 0.3 1" />
    <material name="site_mat" rgba="0.95 0.9 0.15 0.9" />
    <material name="rail_limit_mat" rgba="0.95 0.55 0.05 1" />
  </asset>
  <worldbody>
    <camera name="spectate" pos="0 3 1.4" fovy="90" xyaxes="-1 0 0 0 -0.1240 0.9923" />
    <light diffuse="0.7 0.7 0.7" pos="0 0 3.5" dir="0 0 -1" />
    <geom name="floor" type="plane" size="5 5 0.1" material="floor_mat" />
    <body name="frame">
      <geom name="rail" type="box" pos="0 0 0.85" size="2.2 0.05 0.05" material="metal_mat" />
      <geom name="rail_limit_left" type="box" pos="-2.2 0 0.95" size="0.025 0.12 0.1" material="rail_limit_mat" />
      <geom name="rail_limit_right" type="box" pos="2.2 0 0.95" size="0.025 0.12 0.1" material="rail_limit_mat" />
    </body>
    <body name="cart" pos="0 0 1">
      <joint name="cart_slide" type="slide" axis="1 0 0" range="-2.2 2.2" frictionloss="0.02" damping="0.1" />
      <inertial pos="0 0 0" mass="2" diaginertia="0.0333 0.0333 0.0333" />
      <geom name="cart_geom" type="box" size="0.125 0.08 0.1" mass="2.0" material="cart_mat" />
      <site name="cart_center_site" pos="0 0 0" size="0.02" type="sphere" material="site_mat" />
      <geom name="mount_pin" type="capsule" pos="0 0.095 0" axisangle="1 0 0 1.5708" size="0.015 0.03" material="metal_mat" />
      <body name="pole" pos="0 0.11 0" quat="6.12323399574e-17 0 -1 0">
        <joint name="pole_hinge" type="hinge" axis="0 -1 0" frictionloss="0.01" damping="0.03" ref="3.1415926535897931" limited="false" />
        <inertial pos="0 0 0.3" mass="0.7" diaginertia="0.0210233333333 0.0210933333333 0.000116666666667" />
        <site name="pole_hinge_site" pos="0 0 0" size="0.015" type="sphere" material="site_mat" />
        <geom name="pole_geom" type="box" pos="0 0 0.3" size="0.02 0.01 0.3" mass="0.7" material="pole_mat" />
        <site name="pole_tip_site" pos="0 0 0.6" size="0.015" type="sphere" material="site_mat" />
      </body>
    </body>
    <camera name="replay" pos="0 6 1.4" fovy="50" xyaxes="-1 0 0 0 -0.15 0.988686" />
  </worldbody>
  <sensor>
    <jointpos name="cart_position" joint="cart_slide" />
    <jointpos name="pole1_relative_angle" joint="pole_hinge" />
    <jointvel name="cart_velocity" joint="cart_slide" />
    <jointvel name="pole1_relative_velocity" joint="pole_hinge" />
  </sensor>
  <actuator>
    <motor name="cart_motor" joint="cart_slide" gear="1" ctrlrange="-100 100" forcerange="-100 100" />
  </actuator>
</mujoco>
"""
model=mujoco.MjModel.from_xml_string(MODEL_XML); data=mujoco.MjData(model)
def angle(theta): return float((theta+np.pi)%(2*np.pi)-np.pi)
def state(): return np.array([data.qpos[0],angle(data.qpos[1]),data.qvel[0],data.qvel[1]],float)


## 3. True mechanical-energy swing-up

With upright angle zero, pole energy is `0.5*J*dtheta^2 + m*g*l*cos(theta)` and
desired energy is `m*g*l`. The controller drives their difference toward zero while
centering the cart. A small deterministic kick breaks the exact hanging equilibrium.
This cell is independent of PID and LQR.


In [ ]:
class EnergySwingUp:
    def __init__(self):
        self.l=POLE_LENGTH/2; self.J=POLE_MASS*POLE_LENGTH**2/3
        self.desired=POLE_MASS*GRAVITY*self.l
    def energy(self,theta,dtheta): return .5*self.J*dtheta**2+POLE_MASS*GRAVITY*self.l*np.cos(theta)
    def command(self,x,theta,dx,dtheta):
        error=self.energy(theta,dtheta)-self.desired
        pump=-ENERGY_GAIN*error*dtheta*np.cos(theta)
        if abs(dtheta)<.05 and abs(theta)>2.8: pump=STARTUP_FORCE*np.sign(theta if theta else 1.)
        rail_scale=max(.05,1-(abs(x)/RAIL_LIMIT)**4)
        force=rail_scale*pump-CENTER_KP*x-CENTER_KD*dx
        if abs(x) > .82*RAIL_LIMIT:
            force=-np.sign(x)*SWING_LIMIT-CENTER_KD*dx
        return float(np.clip(force,-SWING_LIMIT,SWING_LIMIT)),float(error)


## 4. PID upright stabilizer
A cascaded cart-position loop requests a small pole tilt; the angle loop produces force.


In [ ]:
class PIDStabilizer:
    def __init__(self): self.reset()
    def reset(self): self.position_integral=self.angle_integral=0.
    def command(self,x,theta,dx,dtheta,target_x=0.):
        g=PID_GAINS; position_error=target_x-x
        self.position_integral=np.clip(self.position_integral+position_error*DT,-1,1)
        tilt=-(g["position_kp"]*position_error+g["position_ki"]*self.position_integral-g["position_kd"]*dx)
        tilt=np.clip(tilt,-g["max_tilt"],g["max_tilt"])
        angle_error=theta-tilt; self.angle_integral=np.clip(self.angle_integral+angle_error*DT,-.5,.5)
        force=-(g["angle_kp"]*angle_error+g["angle_ki"]*self.angle_integral+g["angle_kd"]*dtheta)
        return float(np.clip(force,-ACTUATOR_LIMIT,ACTUATOR_LIMIT))


## 5. LQR upright stabilizer
The notebook numerically linearizes the exact embedded MuJoCo plant, then solves CARE.


In [ ]:
linear_model=mujoco.MjModel.from_xml_string(MODEL_XML)
linear_model.dof_frictionloss[:]=0.0  # CARE requires a smooth local model.
def acceleration(current_state,force):
    probe=mujoco.MjData(linear_model); probe.qpos[:]=current_state[:2]; probe.qvel[:]=current_state[2:]
    probe.ctrl[0]=force; mujoco.mj_forward(linear_model,probe)
    return np.array([probe.qvel[0],probe.qvel[1],probe.qacc[0],probe.qacc[1]])

def linearize(eps=1e-5):
    equilibrium=np.zeros(4); A=np.column_stack([(acceleration(equilibrium+np.eye(4)[i]*eps,0)-acceleration(equilibrium-np.eye(4)[i]*eps,0))/(2*eps) for i in range(4)])
    B=((acceleration(equilibrium,eps)-acceleration(equilibrium,-eps))/(2*eps))[:,None]
    return A,B

class LQRStabilizer:
    def __init__(self):
        self.A,self.B=linearize(); P=solve_continuous_are(self.A,self.B,LQR_Q,LQR_R)
        self.K=np.linalg.solve(LQR_R,self.B.T@P)
        assert np.all(np.real(np.linalg.eigvals(self.A-self.B@self.K))<0)
        print("LQR K:",self.K.round(3),"poles:",np.linalg.eigvals(self.A-self.B@self.K).round(3))
    def reset(self): pass
    def command(self,x,theta,dx,dtheta,target_x=0.):
        error=np.array([x-target_x,theta,dx,dtheta])
        return float(np.clip(-(self.K@error)[0],-ACTUATOR_LIMIT,ACTUATOR_LIMIT))


## 6. Choose PID or LQR
Change only `STABILIZER`, rerun from this cell, and compare saved trajectories.


In [ ]:
STABILIZER = "LQR"  # EDIT ME: "PID" or "LQR"
if STABILIZER not in {"PID","LQR"}: raise ValueError("STABILIZER must be PID or LQR")
stabilizer=PIDStabilizer() if STABILIZER=="PID" else LQRStabilizer()
swing=EnergySwingUp()


## 7. Hybrid supervisor

Capture requires angle, pole speed, cart position, and cart speed to remain safe for a
dwell period. Hysteresis returns to swing-up after a large angular departure. During
catch, the cart target moves gradually from capture position back to center.


In [ ]:
class HybridController:
    def __init__(self,swing,stabilizer): self.swing,self.stabilizer=swing,stabilizer; self.reset()
    def reset(self): self.mode="SWING"; self.capture_candidate=None; self.capture_time=None; self.capture_x=0.; self.stabilizer.reset()
    def command(self,t,x,theta,dx,dtheta):
        safe=abs(theta)<CAPTURE_ANGLE and abs(dtheta)<CAPTURE_SPEED and abs(x)<CAPTURE_X and abs(dx)<CAPTURE_DX
        if self.mode=="SWING":
            self.capture_candidate=t if safe and self.capture_candidate is None else self.capture_candidate
            if not safe: self.capture_candidate=None
            if self.capture_candidate is not None and t-self.capture_candidate>=CAPTURE_DWELL:
                self.mode="BALANCE"; self.capture_time=t; self.capture_x=x; self.stabilizer.reset()
        elif abs(theta)>FALLBACK_ANGLE:
            self.mode="SWING"; self.capture_candidate=None
        if self.mode=="SWING":
            force,error=self.swing.command(x,theta,dx,dtheta)
        else:
            blend=min(1.,(t-self.capture_time)/CATCH_SECONDS); target_x=(1-blend)*self.capture_x
            force=self.stabilizer.command(x,theta,dx,dtheta,target_x); error=self.swing.energy(theta,dtheta)-self.swing.desired
        return force,error,self.mode


## 8. Simulate and save
The trial stops at rail contact. Configuration and trajectory are persisted for comparison.


In [ ]:
mujoco.mj_resetData(model,data); data.qpos[:]=[0.,np.pi+.03]; data.qvel[:]=[0.,0.]; mujoco.mj_forward(model,data)
controller=HybridController(swing,stabilizer); rows=[]
for step in range(int(SIMULATION_SECONDS/DT)):
    t=step*DT; x,theta,dx,dtheta=state(); force,error,mode=controller.command(t,x,theta,dx,dtheta)
    data.ctrl[0]=force; mujoco.mj_step(model,data)
    rows.append({"time":t,"x":x,"theta":theta,"dx":dx,"dtheta":dtheta,"force":force,"energy_error":error,"mode":mode})
    if abs(data.qpos[0])>=RAIL_LIMIT: print("Stopped at rail",t); break
trajectory=pd.DataFrame(rows); trajectory.to_csv(TRAJECTORY_PATH,index=False)
configuration={"stabilizer":STABILIZER,"energy_gain":ENERGY_GAIN,"center_kp":CENTER_KP,"center_kd":CENTER_KD,
 "swing_limit":SWING_LIMIT,"actuator_limit":ACTUATOR_LIMIT,"pid_gains":PID_GAINS,
 "lqr_q":np.diag(LQR_Q).tolist(),"lqr_r":float(LQR_R[0,0]),"capture_angle":CAPTURE_ANGLE,
 "capture_speed":CAPTURE_SPEED,"capture_dwell":CAPTURE_DWELL,"fallback_angle":FALLBACK_ANGLE}
CONFIG_PATH.write_text(json.dumps(configuration,indent=2)); print("Saved",TRAJECTORY_PATH,CONFIG_PATH)


## 9. Static dashboard
Successful tuning captures upright, avoids the rail, and remains in BALANCE.


In [ ]:
fig,axes=plt.subplots(3,2,figsize=(15,11),sharex=True,constrained_layout=True)
axes[0,0].plot(trajectory.time,np.degrees(trajectory.theta)); axes[0,0].axhline(0,color="k",alpha=.3); axes[0,0].set_ylabel("Angle (deg)")
axes[0,1].plot(trajectory.time,trajectory.x); axes[0,1].axhline(RAIL_LIMIT,color="r",ls="--"); axes[0,1].axhline(-RAIL_LIMIT,color="r",ls="--"); axes[0,1].set_ylabel("Cart x (m)")
axes[1,0].plot(trajectory.time,trajectory.energy_error); axes[1,0].set_ylabel("Energy error (J)")
axes[1,1].plot(trajectory.time,trajectory.force); axes[1,1].set_ylabel("Force (N)")
axes[2,0].plot(trajectory.time,trajectory.dtheta); axes[2,0].set_ylabel("Pole speed (rad/s)")
mode=(trajectory["mode"]=="BALANCE").astype(int); axes[2,1].step(trajectory.time,mode,where="post"); axes[2,1].set_yticks([0,1],["SWING","BALANCE"])
for axis in axes.flat: axis.grid(alpha=.25); axis.set_xlabel("Time (s)")
fig.suptitle(f"Energy Swing-Up + {STABILIZER}",fontsize=16,fontweight="bold"); fig.savefig(DASHBOARD_PATH,dpi=160); plt.show(); print("Saved",DASHBOARD_PATH)


## Replay The Trained Run

Colab cannot reliably open MuJoCo's interactive desktop viewer. This block runs a
deterministic evaluation, streams rendered frames directly into an MP4, saves it in
the run directory, and displays it inline. It replays the current trained policy or
controller; it is not an exact recording of a stochastic training episode.


In [ ]:
import imageio.v2 as imageio
from IPython.display import Video, display

REPLAY_SECONDS, REPLAY_FPS = SIMULATION_SECONDS, 50
REPLAY_PATH = OUTPUT_DIR / f"replay_{STABILIZER.lower()}.mp4"
data = mujoco.MjData(model)
data.qpos[:] = [0.0, np.pi + 0.03]
mujoco.mj_forward(model, data)
replay_stabilizer = PIDStabilizer() if STABILIZER == "PID" else LQRStabilizer()
replay_controller = HybridController(EnergySwingUp(), replay_stabilizer)
renderer = mujoco.Renderer(model, height=480, width=640)
writer = imageio.get_writer(REPLAY_PATH, fps=REPLAY_FPS, codec="libx264", quality=8)
frame_stride = max(1, round(1 / (DT * REPLAY_FPS)))
try:
    for step in range(int(REPLAY_SECONDS / DT)):
        t = step * DT
        x, theta, dx, dtheta = state()
        force, _, _ = replay_controller.command(t, x, theta, dx, dtheta)
        data.ctrl[0] = force
        mujoco.mj_step(model, data)
        if step % frame_stride == 0:
            renderer.update_scene(data, camera="replay")
            writer.append_data(renderer.render())
        if abs(data.qpos[0]) >= RAIL_LIMIT:
            break
finally:
    writer.close()
    renderer.close()
print("Saved replay:", REPLAY_PATH)
display(Video(str(REPLAY_PATH), embed=True))
